# OTTO – Multi-Objective Recommender System

**Goal.** Given the events of an e-commerce session so far (clicks, cart additions, orders), predict what the user does next. For every session we predict **three lists of 20 items**: the next *click*, the *carts* and the *orders*.

**Score.** Recall@20 per target, combined as `0.10 × clicks + 0.30 × carts + 0.60 × orders` (orders matter most).

**What this notebook does (run the cells from top to bottom):**
1. Load the data and explore it (graphs)
2. Build a time-based validation set that imitates the competition
3. **Baseline (v1):** co-visitation matrices + simple rules
4. **Ranker (v2):** candidate generation, features, item2vec, and **LightGBM training** (with training curves)
5. Compare the results and look at the Kaggle leaderboard scores
6. Conclusions

The code lives in the `src/` folder of the project; this notebook calls it step by step so every stage can be seen.

## 0. Setup

In [ ]:
import os, sys, time, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# make sure we run from the project root so that `src` and `data/` are found
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.data import load, make_validation
from src.ranker import (histories, train_item2vec, no_embeddings, build_resources, gen_batch,
                        make_labels, positive_groups, top20, candidate_recall, MATRIX_FOR, FEATURES)
from src.recommend import recommend
from src.evaluate import score, WEIGHTS

plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False})
TARGETS = ["clicks", "carts", "orders"]
EVENT_NAMES = {0: "clicks", 1: "carts", 2: "orders"}
print("project root:", ROOT)

### Settings
`SAMPLE` is the share of sessions used (the full dataset has 12M sessions and does not fit in a laptop's memory).
- `SAMPLE = 0.02` quick demo (a few minutes)
- `SAMPLE = 0.05` the setting used for the results in the report (about 15 minutes)

The co-visitation tables are slow to build, so they are cached in `data/` (set `USE_CACHE = False` to rebuild them).

In [ ]:
TRAIN_PARQUET = "data/train.parquet"   # created by:  python -m src.run convert --src data/train.jsonl --dst data/train.parquet
SAMPLE = 0.05          # share of sessions to use
VAL_DAYS = 7           # the last 7 days become the validation period
MAX_SESSIONS = 40000   # sessions the LightGBM rankers learn from
N_ESTIMATORS = 300     # maximum number of trees per ranker
USE_CACHE = True

## 1. Data

In [ ]:
df = load(TRAIN_PARQUET, SAMPLE)
print(f"{len(df):,} events | {df.session.nunique():,} sessions | {df.aid.nunique():,} distinct items")
df.head()

Each row is one event: the `session`, the item (`aid`), the timestamp in seconds (`ts`) and the event `type` (0 = click, 1 = cart, 2 = order).

### 1.1 Events by type

In [ ]:
counts = df.type.value_counts().sort_index()
labels = [EVENT_NAMES[i] for i in counts.index]
fig, ax = plt.subplots()
bars = ax.bar(labels, counts.values, color=["#4C78A8", "#F58518", "#54A24B"])
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v / counts.sum():.1%}", ha="center", va="bottom")
ax.set_ylabel("number of events")
ax.set_title("Events by type")
plt.show()

Clicks dominate; carts and orders are rare. This is why predicting orders is hard even though it carries 60% of the score.

### 1.2 Session length

In [ ]:
lens = df.groupby("session").size()
fig, ax = plt.subplots()
ax.hist(lens.clip(upper=60), bins=np.arange(1, 62), color="#4C78A8")
ax.set_yscale("log")
ax.set_xlabel("events per session (clipped at 60)")
ax.set_ylabel("number of sessions (log scale)")
ax.set_title("Most sessions are very short")
plt.show()
print(f"median length: {lens.median():.0f} | mean: {lens.mean():.1f} | max: {lens.max()}")

### 1.3 Item popularity

In [ ]:
pop = df.aid.value_counts().to_numpy()
fig, ax = plt.subplots()
ax.loglog(np.arange(1, len(pop) + 1), pop, color="#4C78A8")
ax.set_xlabel("item rank (1 = most popular)")
ax.set_ylabel("number of events")
ax.set_title("Long-tail item popularity (log-log)")
plt.show()
top1 = pop[: max(1, int(0.01 * len(pop)))].sum() / pop.sum()
print(f"the top 1% of items get {top1:.1%} of all events")

A few items are very popular and most items are rare. Popular items alone are a weak recommender, which is why we need session-based signals.

### 1.4 Activity over time

In [ ]:
day = pd.to_datetime(df.ts, unit="s").dt.floor("D")
ct = pd.crosstab(day, df.type.map(EVENT_NAMES))[["clicks", "carts", "orders"]]
ct.index = ct.index.strftime("%d %b")
ax = ct.plot.bar(stacked=True, figsize=(10, 4), color=["#4C78A8", "#F58518", "#54A24B"], width=0.85)
ax.set_xlabel("")
ax.set_ylabel("events per day")
ax.set_title("Events per day (the data covers about four weeks)")
ax.legend(title="type", loc="upper left", bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=60)
plt.show()

### 1.5 What share of sessions reach each step?

In [ ]:
share = df.groupby("type").session.nunique() / df.session.nunique()
fig, ax = plt.subplots()
ax.bar([EVENT_NAMES[i] for i in share.index], share.values, color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_ylabel("share of sessions with at least one such event")
ax.set_title("Funnel: click, then cart, then order")
for i, v in enumerate(share.values):
    ax.text(i, v, f"{v:.1%}", ha="center", va="bottom")
plt.show()

## 2. Validation set (imitating the competition)

The real test sessions are **cut at some point in time** and we must predict what happens after the cut. We reproduce this on our own data:
- the **last 7 days** become validation sessions, each cut at a random point
- the events **before the cut** are the input (*history*)
- the events **after the cut** are the truth: the first click, and all later carts and orders
- everything we learn (co-visitation, popularity, item2vec) uses only **earlier** sessions, so nothing leaks from the future

In [ ]:
train, history, gt = make_validation(df, VAL_DAYS)
print(f"base events (used to learn statistics): {len(train):,}")
print(f"validation history events (the input) : {len(history):,}")
print({t: len(gt[t]) for t in TARGETS}, "<- validation sessions that have a true item per target")

One example: what the model sees and what it must predict.

In [ ]:
hs_all = histories(history)
example = next(h for h in hs_all if h[0] in gt["carts"] and len(h[1]) >= 4)
sid, aids, types, ts = example
print(f"session {sid}")
print("history (input):", [(a, EVENT_NAMES[t]) for a, t in zip(aids, types)])
for t in TARGETS:
    print(f"truth for {t:6s}:", sorted(gt[t].get(sid, [])))

## 3. Baseline (v1): co-visitation

**Co-visitation:** two items are *neighbours* if users often view them in the same session. For every item we keep its 20 best neighbours. We build two tables:
- `recency`: recent events count more (used for clicks)
- `type`: carts and orders count more than clicks (used for carts and orders)

This takes a while the first time (the result is cached on disk). Item2vec embeddings (word2vec on sessions, used later as a feature) are trained here as well.

In [ ]:
t0 = time.time()
cache = Path(f"data/cache_resources_s{SAMPLE}_d{VAL_DAYS}.pkl")
if USE_CACHE and cache.exists():
    with open(cache, "rb") as f:
        r = pickle.load(f)
    print(f"loaded cached resources from {cache}")
else:
    emb = train_item2vec(train) or no_embeddings()
    r = build_resources(train, emb)
    if USE_CACHE:
        with open(cache, "wb") as f:
            pickle.dump(r, f, protocol=5)
print(f"done in {time.time() - t0:.0f}s | items with neighbours: {len(r.cv['type']):,}")

A look at what a co-visitation row looks like: the neighbours of one popular item, with their weights.

In [ ]:
item = r.popular[0]
print(f"item {item}: top neighbours (item, weight)")
print("  recency:", [(b, round(w, 1)) for b, w in r.cv['recency'][item][:5]])
print("  type   :", [(b, round(w, 1)) for b, w in r.cv['type'][item][:5]])

### 3.1 Item2vec: does the embedding capture similar items?

In [ ]:
E, idx = r.E, r.emb_idx
has_emb = E.shape[0] > 1
if has_emb:
    item = r.popular[0]
    sims = E[:-1] @ E[idx[item]]                              # cosine similarity (vectors are normalised)
    keys = np.flatnonzero(idx < E.shape[0] - 1)               # item ids that have an embedding
    order = np.argsort(-sims[idx[keys]])[1:6]                 # [0] is the item itself
    print(f"items most similar to {item} (cosine):", [(int(keys[i]), round(float(sims[idx[keys[i]]]), 2)) for i in order])

    # 2D picture of the embedding space (PCA) for the 3000 most popular items
    top = np.array([a for a in r.popular[:100]] + list(df.aid.value_counts().index[:3000]))
    top = np.unique(top[idx[top] < E.shape[0] - 1])[:3000]
    M = E[idx[top]]
    M = M - M.mean(0)
    U, S, Vt = np.linalg.svd(M, full_matrices=False)
    P = M @ Vt[:2].T
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(P[:, 0], P[:, 1], s=4, alpha=0.5)
    ax.set_title("Item2vec embeddings of popular items (PCA, 2D)")
    ax.set_xlabel("component 1"); ax.set_ylabel("component 2")
    plt.show()
else:
    print("item2vec is switched off (gensim is not installed); the embedding features will be 0.")

### 3.2 Baseline predictions and score

For one session the baseline fills the 20 slots as follows: the session's own items (weighted by recency and event type: clicks 1, carts 6, orders 3), then the co-visitation neighbours of the recent items, then popular items.

In [ ]:
hs = histories(history)
np.random.default_rng(0).shuffle(hs)
val_hs = [h for h in hs if h[0] % 5 == 0][: max(5000, MAX_SESSIONS // 4)]
tr_hs = [h for h in hs if h[0] % 5 != 0][:MAX_SESSIONS]
va_set = {h[0] for h in val_hs}
gt_val = {t: {s: v for s, v in gt[t].items() if s in va_set} for t in TARGETS}
print(f"{len(tr_hs):,} sessions for training the rankers | {len(val_hs):,} sessions for validation")

base_preds = {t: {h[0]: recommend(h[1], h[2], r.cv[MATRIX_FOR[t]], r.popular) for h in val_hs} for t in TARGETS}
base_per, base_joint = score(base_preds, gt_val)
print({t: round(v, 4) for t, v in base_per.items()}, "| joint:", round(base_joint, 4))

In [ ]:
fig, ax = plt.subplots()
vals = [base_per[t] for t in TARGETS] + [base_joint]
names = TARGETS + ["JOINT"]
bars = ax.bar(names, vals, color=["#4C78A8", "#4C78A8", "#4C78A8", "#E45756"])
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.3f}", ha="center", va="bottom")
ax.set_ylabel("recall@20")
ax.set_title("Baseline (v1): validation recall@20")
plt.show()

## 4. Ranker (v2): candidates, features and LightGBM

**Idea.** The baseline uses fixed rules. v2 learns the rules:
1. **Candidates:** for each session, collect up to 100 plausible items (the session's own items, co-visitation neighbours, popular items).
2. **Features:** for every (session, candidate) pair compute numbers such as: how recently and how often the user touched the item, the co-visitation scores, the item2vec similarity to the last item, the item's global popularity, the session length.
3. **Label:** a candidate gets label 1 if the user really did that action on it after the cut, otherwise 0.
4. **Model:** **LightGBM** (gradient-boosted trees, `lambdarank` objective) learns to put the right items at the top. One model per target.

### 4.1 Candidates and features

In [ ]:
t0 = time.time()
tr_sids, tr_sz, tr_aids, tr_X = gen_batch(tr_hs, r, every=0)
va_sids, va_sz, va_aids, va_X = gen_batch(val_hs, r, every=0)
print(f"{time.time() - t0:.0f}s | training rows: {len(tr_aids):,} | validation rows: {len(va_aids):,} | features: {tr_X.shape[1]}")
print(f"average candidates per session: {tr_sz.mean():.0f}")

The first rows of the training table for one session (candidate item + its features):

In [ ]:
k = tr_sz[0]
view = pd.DataFrame(tr_X[:k], columns=FEATURES)
view.insert(0, "aid", tr_aids[:k])
view.head(8).round(2)

**Candidate recall** = how many of the true items are somewhere in the candidate list. A ranker can only reorder what is in the list, so this is an upper limit for the ranker.

In [ ]:
cand_rec = {t: candidate_recall(va_sids, va_sz, va_aids, gt_val[t]) for t in TARGETS}
print({t: round(v, 4) for t, v in cand_rec.items()})

### 4.2 Training LightGBM

The cell below **trains one ranker per target** and prints the validation NDCG@20 every 20 trees. Training stops early when the validation score has not improved for 30 trees. (Sessions without any correct candidate are skipped, because they teach nothing.)

In [ ]:
import lightgbm as lgb

models, curves = {}, {}
for t in TARGETS:
    ytr = make_labels(tr_sids, tr_sz, tr_aids, gt[t])
    yva = make_labels(va_sids, va_sz, va_aids, gt[t])
    Xt, yt, gtr = positive_groups(tr_X, ytr, tr_sz)
    Xv, yv, gva = positive_groups(va_X, yva, va_sz)
    print(f"\n===== target: {t} | training sessions: {len(gtr):,} | rows: {len(yt):,} | positives: {int(yt.sum()):,}")

    evals = {}
    model = lgb.LGBMRanker(objective="lambdarank", n_estimators=N_ESTIMATORS, learning_rate=0.05,
                           num_leaves=63, min_child_samples=50, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, n_jobs=-1, verbose=-1)
    model.fit(Xt, yt, group=gtr,
              eval_set=[(Xt, yt), (Xv, yv)], eval_group=[gtr, gva], eval_names=["train", "valid"], eval_at=[20],
              callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(20), lgb.record_evaluation(evals)])
    models[t], curves[t] = model, evals
    print(f"stopped at {model.best_iteration_ or N_ESTIMATORS} trees")

### 4.3 Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, t in zip(axes, TARGETS):
    ev = curves[t]
    ax.plot(ev["train"]["ndcg@20"], label="train")
    ax.plot(ev["valid"]["ndcg@20"], label="validation")
    best = models[t].best_iteration_
    if best:
        ax.axvline(best - 1, color="grey", ls="--", label=f"best = {best}")
    ax.set_title(f"{t}")
    ax.set_xlabel("number of trees")
    ax.set_ylabel("NDCG@20")
    ax.legend()
fig.suptitle("LightGBM training curves", y=1.02)
plt.tight_layout()
plt.show()

The training curve keeps rising while the validation curve flattens: the model starts to memorise the training sessions, which is why early stopping is used.

### 4.4 Which features does the model use?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, t in zip(axes, TARGETS):
    gain = models[t].booster_.feature_importance(importance_type="gain")
    order = np.argsort(gain)
    ax.barh(np.array(FEATURES)[order], gain[order] / gain.sum(), color="#4C78A8")
    ax.set_title(f"feature importance: {t}")
    ax.set_xlabel("share of total gain")
plt.tight_layout()
plt.show()

### 4.5 Predictions and score

In [ ]:
v2_preds = {t: top20(models[t], va_sids, va_sz, va_aids, va_X) for t in TARGETS}
v2_per, v2_joint = score(v2_preds, gt_val)

result = pd.DataFrame({
    "candidate recall": cand_rec,
    "baseline R@20": base_per,
    "v2 R@20": v2_per,
}).T[TARGETS]
result["JOINT"] = [np.nan, base_joint, v2_joint]
result.round(4)

## 5. Results

### 5.1 Baseline vs v2 (local validation)

In [ ]:
names = TARGETS + ["JOINT"]
base_vals = [base_per[t] for t in TARGETS] + [base_joint]
v2_vals = [v2_per[t] for t in TARGETS] + [v2_joint]
cand_vals = [cand_rec[t] for t in TARGETS] + [np.nan]

x = np.arange(len(names)); w = 0.27
fig, ax = plt.subplots(figsize=(9, 4.5))
b1 = ax.bar(x - w, base_vals, w, label="baseline (v1)", color="#4C78A8")
b2 = ax.bar(x, v2_vals, w, label="LightGBM ranker (v2)", color="#F58518")
b3 = ax.bar(x + w, cand_vals, w, label="candidate recall (upper limit for a ranker)", color="#BBBBBB")
for bars in (b1, b2, b3):
    for b in bars:
        h = b.get_height()
        if not np.isnan(h):
            ax.text(b.get_x() + b.get_width() / 2, h, f"{h:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel("recall@20")
ax.set_title("Validation: baseline vs LightGBM ranker")
ax.set_ylim(0, np.nanmax(base_vals + v2_vals + cand_vals) * 1.3)
ax.legend(loc="upper center", ncol=3, fontsize=8, frameon=False)
plt.show()

**Reading the chart (real-data runs).** In our runs on the real data the ranker and the baseline ended up very close (see the conclusions for the numbers); the exact values on a fresh run will differ a little. The grey bars show why: even if the ranker ordered the 100 candidates perfectly, there is limited room above the baseline, because the correct item is often **not in the candidate list at all**. The bottleneck is candidate generation, not the ranking model.

### 5.2 Kaggle leaderboard (late submissions)

These scores come from real Kaggle submissions of the baseline (the competition ended in January 2023, so they are late submissions and do not count in the official ranking). The v2 ranker was only evaluated locally because it did not beat the baseline.

In [ ]:
kaggle = pd.DataFrame({
    "submission": ["v1: co-visitation, 10% of train", "v1b: co-visitation, 20% of train"],
    "public": [0.55687, 0.56156],
    "private": [0.55776, 0.56189],
})
x = np.arange(len(kaggle)); w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
b1 = ax.bar(x - w / 2, kaggle.public, w, label="public score", color="#4C78A8")
b2 = ax.bar(x + w / 2, kaggle.private, w, label="private score", color="#F58518")
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{b.get_height():.5f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(kaggle.submission, fontsize=8)
ax.set_ylim(0.54, 0.57)
ax.set_ylabel("score (joint recall@20)")
ax.set_title("Kaggle scores of the baseline: more data helps a little")
ax.legend()
plt.show()
kaggle

## 6. Conclusions

- **Problem.** Predict clicks, carts and orders from a user session; the score is a weighted recall@20 in which orders count 60%.
- **Method.** A two-stage system: candidate generation (co-visitation tables, popular items, item2vec as a feature), then a LightGBM ranker per target.
- **Baseline result.** The co-visitation baseline scored **0.55687** (10% of the training sessions) and **0.56156** (20%) on the Kaggle public leaderboard (late submission). Doubling the data gave only about +0.005.
- **LightGBM result.** In our local validation, the ranker did **not** beat the baseline (for example 0.5341 vs 0.5358 joint recall on a 5% sample, and 0.5298 vs 0.5298 with four times more training sessions). More training data did not change this.
- **Why.** The candidate lists are the limit: the true item is often missing from them, and a ranker cannot recover items it never sees.
- **Next steps.** Richer candidate generation: more neighbours per item, matrices built only from carts and orders, a larger candidate list (200), and item2vec neighbours as an extra candidate source. A sequence model (GRU or Transformer) could be added later.